In [ ]:
import torch
!pip install torch torchvision

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install torch_geometric

In [ ]:
import pandas as pd
import numpy as np
import torch

from torch_geometric.data import Data

# =========================
# STEP 1: LOAD DATA
# =========================
interactions = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/synchronized_interaction_logs.csv")          # user_id, question_id, timestamp, correct
questions = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/CDL_150_Questions_With_ID.xlsx")      # question_id, skill_id
googlequestions = pd.read_json("/content/drive/MyDrive/Colab Notebooks/google_cloud_quiz_eval.json")      # question_id, skill_id
studentquiz = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/student_quiz_dataset.csv")      # user_id, skill_id, pre_score, post_score

# Rename 'Question_ID' to 'question_id' in the questions DataFrame to match interactions
questions = questions.rename(columns={'Question_ID': 'question_id'})

# Assign 'Module' as 'skill_id' in the questions DataFrame, assuming Module represents the skill.
# If 'skill_id' is located in a different column, please adjust this line accordingly.
questions['skill_id'] = questions['Module']

# Rename 'student_id' to 'user_id' in interactions DataFrame
interactions = interactions.rename(columns={'student_id': 'user_id'})

# Rename 'student_id' to 'user_id' in studentquiz for merging
studentquiz = studentquiz.rename(columns={'student_id': 'user_id'})

# Merge interactions with studentquiz to get the 'status' (correctness)
# Use a left merge to keep all interactions and add status if available
interactions = interactions.merge(studentquiz[['user_id', 'question_id', 'status']],
                                  on=['user_id', 'question_id'],
                                  how='left')

# Convert 'status' to numerical 'correct' column (1 for 'correct', 0 for 'incorrect')
# Fill NaN values (for interactions without a matching status) with 0, marking them as incorrect by default
interactions['correct'] = interactions['status'].map({'correct': 1, 'incorrect': 0}).fillna(0).astype(int)

# Drop the original 'status' column as it's no longer needed
interactions = interactions.drop(columns=['status'], errors='ignore')

# =========================
# STEP 2: MERGE INTERACTIONS WITH SKILLS
# =========================
data = interactions.merge(questions, on="question_id", how="inner")
data = data.sort_values(by=["user_id", "timestamp"])

# Encode IDs
user2idx = {u: i for i, u in enumerate(data["user_id"].unique())}
q2idx = {q: i for i, q in enumerate(data["question_id"].unique())}
s2idx = {s: i for i, s in enumerate(data["skill_id"].unique())}

data["user_id"] = data["user_id"].map(user2idx)
data["question_id"] = data["question_id"].map(q2idx)
data["skill_id"] = data["skill_id"].map(s2idx)

num_users = len(user2idx)
num_skills = len(s2idx)

# =========================
# STEP 3: BUILD STUDENT SEQUENCES
# =========================
student_sequences = []
grouped = data.groupby("user_id")

for user, group in grouped:
    student_sequences.append({
        "student_id": user,
        "q_seq": torch.tensor(group["question_id"].values, dtype=torch.long),
        "s_seq": torch.tensor(group["skill_id"].values, dtype=torch.long),
        "r_seq": torch.tensor(group["correct"].values, dtype=torch.float),
        "latency_seq": torch.tensor(group["elapsed_time"].values, dtype=torch.float) # Added latency_seq
    })

# =========================
# STEP 4: BUILD SKILL GRAPH
# =========================
adj_matrix = np.zeros((num_skills, num_skills))
for seq in student_sequences:
    skills = seq["s_seq"].numpy()
    for i in range(len(skills)-1):
        s1, s2 = skills[i], skills[i+1]
        adj_matrix[s1, s2] += 1
        adj_matrix[s2, s1] += 1

# Convert to PyG edge_index
edges = np.array(np.nonzero(adj_matrix))
edge_index = torch.tensor(edges, dtype=torch.long)
edge_weight = torch.tensor(adj_matrix[edges[0], edges[1]], dtype=torch.float)

# Node features: one-hot for skills
x = torch.eye(num_skills, dtype=torch.float)

# =========================
# STEP 5: CREATE PyG DATA OBJECT
# =========================
data_gnn = Data(x=x, edge_index=edge_index, edge_attr=edge_weight)
data_gnn.student_sequences = student_sequences

# =========================
# STEP 6: SAVE DATA
# =========================
torch.save(data_gnn, "kt_all_models_prepost.pt")
print("Preprocessing complete. Dataset ready for all models (AKT, DKT, BKT, GRKT/GNN).")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from sklearn.metrics import roc_auc_score, mean_squared_error
import numpy as np

# Basic implementation of SAM (Sharpness-Aware Minimization)
class SAM(torch.optim.Optimizer):
    def __init__(self, params, base_optimizer, rho=0.05, adaptive=False, **kwargs):
        assert rho >= 0.0, f"Invalid rho, should be non-negative: {rho}"
        defaults = dict(rho=rho, adaptive=adaptive, **kwargs)
        super(SAM, self).__init__(params, defaults)
        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups
        self.defaults.update(self.base_optimizer.defaults)

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group["rho"] / (grad_norm + 1e-12)
            for p in group["params"]:
                if p.grad is None: continue
                self.state[p]["old_p"] = p.data.clone()
                e_w = (torch.pow(p, 2) if group["adaptive"] else 1.0) * p.grad * scale.to(p)
                p.add_(e_w)  # climb to the local maximum "w + e(w)"
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None: continue
                p.data = self.state[p]["old_p"]  # get back to "w" from "w + e(w)"
        self.base_optimizer.step()  # do the actual "sharpness-aware" update
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def step(self, closure=None):
        assert closure is not None, "Sharpness Aware Minimization requires closure, but it was not provided"
        closure = torch.enable_grad()(closure)
        self.first_step(zero_grad=True)
        closure()
        self.second_step()

    def _grad_norm(self):
        shared_device = self.param_groups[0]["params"][0].device  # put everything on the same device, in case of model parallelism
        norm = torch.norm(
                    torch.stack([
                        ((torch.abs(p) if group["adaptive"] else 1.0) * p.grad).norm(p=2).to(shared_device)
                        for group in self.param_groups for p in group["params"]
                        if p.grad is not None
                    ]),
                    p=2
               )
        return norm

    def load_state_dict(self, state_dict):
        super().load_state_dict(state_dict)
        self.base_optimizer.param_groups = self.param_groups


# AKT Model components
class SelfAttention(nn.Module):
    def __init__(self, hidden_dim, heads):
        super(SelfAttention, self).__init__()
        self.hidden_dim = hidden_dim
        self.heads = heads
        self.head_dim = hidden_dim // heads

        self.q_linear = nn.Linear(hidden_dim, hidden_dim)
        self.k_linear = nn.Linear(hidden_dim, hidden_dim)
        self.v_linear = nn.Linear(hidden_dim, hidden_dim)
        self.out_linear = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)

        q = self.q_linear(q).view(batch_size, -1, self.heads, self.head_dim).transpose(1, 2)
        k = self.k_linear(k).view(batch_size, -1, self.heads, self.head_dim).transpose(1, 2)
        v = self.v_linear(v).view(batch_size, -1, self.heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        attention = torch.softmax(scores, dim=-1)
        out = torch.matmul(attention, v)

        out = out.transpose(1, 2).contiguous().view(batch_size, -1, self.hidden_dim)
        return self.out_linear(out)

class AKT(nn.Module):
    def __init__(self, num_questions, num_skills, hidden_dim, num_blocks, heads, dropout_rate=0.5):
        super(AKT, self).__init__()
        self.num_questions = num_questions
        self.num_skills = num_skills
        self.hidden_dim = hidden_dim

        # Embeddings
        self.q_embed = nn.Embedding(num_questions + 1, hidden_dim)
        self.s_embed = nn.Embedding(num_skills + 1, hidden_dim)
        self.qa_embed = nn.Embedding(2 * num_skills + 1, hidden_dim) # skill_id + correct * num_skills

        self.blocks = nn.ModuleList([
            SelfAttention(hidden_dim, heads) for _ in range(num_blocks)
        ])

        self.dropout = nn.Dropout(dropout_rate)
        self.layer_norm = nn.LayerNorm(hidden_dim)

        self.predict = nn.Linear(hidden_dim, 1)

    def forward(self, q_seq, s_seq, a_seq):
        # q_seq: question sequence, s_seq: skill sequence, a_seq: correctness sequence
        batch_size, seq_len = q_seq.size()

        # Prepare inputs
        q_emb = self.q_embed(q_seq)
        s_emb = self.s_embed(s_seq)

        # Interaction embedding (skill + correctness)
        qa_id = s_seq + a_seq * self.num_skills
        qa_emb = self.qa_embed(qa_id)

        # Combine query/key representations
        x = q_emb + s_emb

        # Create causal mask
        mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0).unsqueeze(0).to(q_seq.device)

        # Attention blocks
        for block in self.blocks:
            residual = x
            x = block(x, qa_emb, qa_emb, mask)
            x = self.dropout(x)
            x = self.layer_norm(x + residual)

        output = torch.sigmoid(self.predict(x).squeeze(-1))
        return output


# Helper function for evaluation
def evaluate_model(model, dataloader, device):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for q_seq, s_seq, a_seq in dataloader:
            q_seq, s_seq, a_seq = q_seq.to(device), s_seq.to(device), a_seq.to(device)

            # Predict for next step based on current sequence
            # In standard KT, we predict a_t based on interaction up to t-1
            # Here we just get the outputs
            outputs = model(q_seq, s_seq, a_seq)

            # Flatten and filter out padding if necessary
            # Assuming valid mask exists, adapt as needed for actual dataloader
            y_pred.extend(outputs.cpu().numpy().flatten())
            y_true.extend(a_seq.cpu().numpy().flatten())

    auc = roc_auc_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return auc, rmse


In [ ]:
!pip install optuna

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

# Load the preprocessed data
data_gnn = torch.load('kt_all_models_prepost.pt', weights_only=False)
student_sequences = data_gnn.student_sequences

class KTDataset(Dataset):
    def __init__(self, sequences, max_seq_len=200):
        self.sequences = sequences
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        q_seq = seq['q_seq']
        s_seq = seq['s_seq']
        r_seq = seq['r_seq']
        latency_seq = seq['latency_seq'] # Get latency sequence

        # Padding or truncating sequences
        seq_len = len(q_seq)
        if seq_len > self.max_seq_len:
            q_seq = q_seq[-self.max_seq_len:]
            s_seq = s_seq[-self.max_seq_len:]
            r_seq = r_seq[-self.max_seq_len:]
            latency_seq = latency_seq[-self.max_seq_len:] # Truncate latency
        elif seq_len < self.max_seq_len:
            pad_len = self.max_seq_len - seq_len
            # Using 0 as padding (assuming indices start at 1 or 0 is handled)
            q_seq = torch.cat([torch.zeros(pad_len, dtype=torch.long), q_seq])
            s_seq = torch.cat([torch.zeros(pad_len, dtype=torch.long), s_seq])
            r_seq = torch.cat([torch.zeros(pad_len, dtype=torch.float), r_seq])
            latency_seq = torch.cat([torch.zeros(pad_len, dtype=torch.float), latency_seq]) # Pad latency

        return q_seq, s_seq, r_seq, latency_seq # Return latency_seq

# Split into train and validation sets (80/20)
train_size = int(0.8 * len(student_sequences))
val_size = len(student_sequences) - train_size
train_seqs, val_seqs = torch.utils.data.random_split(student_sequences, [train_size, val_size])

train_loader = DataLoader(KTDataset(train_seqs), batch_size=32, shuffle=True)
val_loader = DataLoader(KTDataset(val_seqs), batch_size=32, shuffle=False)

# Determine num_questions and num_skills from data_gnn
num_questions = max([seq['q_seq'].max().item() for seq in student_sequences]) + 1
num_skills = max([seq['s_seq'].max().item() for seq in student_sequences]) + 1

print(f"Train sequences: {len(train_seqs)}, Val sequences: {len(val_seqs)}")
print(f"Num questions: {num_questions}, Num skills: {num_skills}")

In [ ]:
import torch.nn as nn
import numpy as np
from sklearn.metrics import roc_auc_score, mean_squared_error
import optuna

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Redefine evaluate_model to include masking for padding
def evaluate_model(model, dataloader, device):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for q_seq, s_seq, a_seq in dataloader:
            q_seq, s_seq, a_seq = q_seq.to(device), s_seq.to(device), a_seq.to(device)

            outputs = model(q_seq, s_seq, a_seq.long())

            # Create a mask to ignore padding (assuming question_id 0 is padding)
            mask = (q_seq != 0)

            # Filter outputs and targets using the mask
            valid_outputs = outputs[mask]
            valid_targets = a_seq[mask]

            y_pred.extend(valid_outputs.cpu().numpy().tolist())
            y_true.extend(valid_targets.cpu().numpy().tolist())

    # Protect against edge case where batch might only have one class
    if len(set(y_true)) > 1:
        auc = roc_auc_score(y_true, y_pred)
    else:
        auc = 0.5
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return auc, rmse

def train_epoch_sam(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for q_seq, s_seq, a_seq in dataloader:
        q_seq, s_seq, a_seq = q_seq.to(device), s_seq.to(device), a_seq.to(device)

        mask = (q_seq != 0)

        # Step 1: SAM first step
        optimizer.zero_grad()
        outputs = model(q_seq, s_seq, a_seq.long())
        # Mask the loss
        loss = criterion(outputs[mask], a_seq[mask])
        loss.backward()
        optimizer.first_step(zero_grad=True)

        # Step 2: SAM second step
        outputs = model(q_seq, s_seq, a_seq.long())
        loss_second = criterion(outputs[mask], a_seq[mask])
        loss_second.backward()
        optimizer.second_step(zero_grad=True)

        total_loss += loss.item()
    return total_loss / len(dataloader)

# Objective function for Optuna
def objective(trial):
    # Hyperparameters to tune
    hidden_dim = trial.suggest_categorical('hidden_dim', [32, 64, 128])
    num_blocks = trial.suggest_int('num_blocks', 1, 3)
    heads = trial.suggest_categorical('heads', [2, 4, 8])
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
    lr = trial.suggest_loguniform('lr', 1e-5, 1e-3)
    rho = trial.suggest_float('rho', 0.01, 0.1)

    model = AKT(num_questions, num_skills, hidden_dim, num_blocks, heads, dropout_rate).to(device)
    base_optimizer = torch.optim.AdamW
    optimizer = SAM(model.parameters(), base_optimizer, rho=rho, lr=lr)
    criterion = nn.BCELoss()

    epochs = 10 # Reduce epochs for faster tuning

    for epoch in range(epochs):
        train_loss = train_epoch_sam(model, train_loader, optimizer, criterion, device)
        val_auc, val_rmse = evaluate_model(model, val_loader, device)

        trial.report(val_auc, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return val_auc


In [ ]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=10) # Reduced n_trials for faster tuning

print("Number of finished trials: ", len(study.trials))
print("Best trial:")
trial = study.best_trial

print(f"  Value: {trial.value:.4f}")
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

# Train final model with best hyperparameters
best_hidden_dim = trial.params['hidden_dim']
best_num_blocks = trial.params['num_blocks']
best_heads = trial.params['heads']
best_dropout_rate = trial.params['dropout_rate']
best_lr = trial.params['lr']
best_rho = trial.params['rho']

final_model = AKT(num_questions, num_skills, best_hidden_dim, best_num_blocks, best_heads, best_dropout_rate).to(device)
final_base_optimizer = torch.optim.AdamW
final_optimizer = SAM(final_model.parameters(), final_base_optimizer, rho=best_rho, lr=best_lr)
final_criterion = nn.BCELoss()

final_epochs = 50 # Train for more epochs after tuning

print(f"\nStarting final training with best hyperparameters for {final_epochs} epochs...")
for epoch in range(final_epochs):
    train_loss = train_epoch_sam(final_model, train_loader, final_optimizer, final_criterion, device)
    if (epoch + 1) % 10 == 0 or epoch == final_epochs - 1:
        val_auc, val_rmse = evaluate_model(final_model, val_loader, device)
        print(f"Epoch {epoch+1:02d}/{final_epochs} | Train Loss: {train_loss:.4f} | Val AUC: {val_auc:.4f} | Val RMSE: {val_rmse:.4f}")

# Evaluate the final tuned model
final_auc, final_rmse = evaluate_model(final_model, val_loader, device)
print(f"\nAKT Model (tuned) AUC: {final_auc:.4f}")
print(f"AKT Model (tuned) RMSE: {final_rmse:.4f}")

In [ ]:
import pandas as pd
import numpy as np

final_model.eval()

all_q_ids = []
all_s_ids = []
all_actual_correctness = []
all_predicted_probabilities = []
all_latencies = []

with torch.no_grad():
    for q_seq_batch, s_seq_batch, r_seq_batch, latency_seq_batch in val_loader:
        q_seq_batch = q_seq_batch.to(device)
        s_seq_batch = s_seq_batch.to(device)
        r_seq_batch = r_seq_batch.to(device)
        latency_seq_batch = latency_seq_batch.to(device)

        outputs = final_model(q_seq_batch, s_seq_batch, r_seq_batch.long())

        # Create a mask to ignore padding (assuming question_id 0 is padding)
        mask = (q_seq_batch != 0)

        all_q_ids.extend(q_seq_batch[mask].cpu().numpy())
        all_s_ids.extend(s_seq_batch[mask].cpu().numpy())
        all_actual_correctness.extend(r_seq_batch[mask].cpu().numpy())
        all_predicted_probabilities.extend(outputs[mask].cpu().numpy())
        all_latencies.extend(latency_seq_batch[mask].cpu().numpy())

results_df = pd.DataFrame({
    'Question ID': all_q_ids,
    'Encoded Skill ID': all_s_ids,
    'Actual Correctness': all_actual_correctness,
    'Predicted Probability (Correct)': all_predicted_probabilities,
    'Latency': all_latencies
})

# Apply standard log scaling to match the small decimal format from your other notebook
# Converts ms to seconds and applies log(x + 1)
results_df['Latency'] = np.log1p(results_df['Latency'] / 1000.0)

# --- Modified section to display for three specified question IDs with custom formatting ---
if not results_df.empty:
    unique_q_ids = results_df['Question ID'].unique()

    # Choose 3 random unique question IDs
    num_q_ids_to_display = 3
    if len(unique_q_ids) < num_q_ids_to_display:
        num_q_ids_to_display = len(unique_q_ids)

    chosen_q_ids = np.random.choice(unique_q_ids, size=num_q_ids_to_display, replace=False)

    print("\nLatency Test Results for 3 Random Question IDs:\n")
    for q_id in chosen_q_ids:
        specified_q_details_df = results_df[results_df['Question ID'] == q_id]
        # Select only the first interaction for this question ID
        if not specified_q_details_df.empty:
            row = specified_q_details_df.iloc[0]
            print(f"Question ID: {int(row['Question ID'])}")
            print(f" Encoded Skill ID: {int(row['Encoded Skill ID'])}")
            print(f" Predicted Probability (Correct): {row['Predicted Probability (Correct)']:.4f}")
            print(f" Latency: {row['Latency']:.4f}")
            print()
else:
    print("No results to display. Please ensure the model has been trained and predictions generated.")